# Line Emission Denoising ΓÇö Full-Image U-Net (Week-5 pivot)

Per the 2026-06-18 mentor pivot: **full-image** denoising of line-emission velocity channels
(NO patches), using **last week's U-Net** (`DenoisingUNet`, NOT the DDPM), trained channel-by-channel.

- Dataset: line-emission FITS cubes, (201, 600, 600), split at the **cube level** (3 RunID groups
  held out for inference only).
- Channels sampled per cube via the Gaussian sampler (center 100, ~75% in [50,150]).
- Each channel downsampled to **256├ù256**, per-channel min-max normalised to [0,1].
- Loss: `HybridLoss(0.8, 0.2)` = 0.8┬╖MSE + 0.2┬╖(1ΓêÆSSIM). Optimizer Adam lr=1e-3.

### Kaggle setup
GPU on, Internet on, *Add Input* ΓåÆ your line-emission Dataset (FITS cubes). The bootstrap finds it
under `/kaggle/input/` and points the split at it.


## 0. Bootstrap (clone repo for src/, locate data)


In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch','line-emission','--depth','1','https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin','line-emission'], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/line-emission'], check=True)
    # pytorch-msssim for the SSIM loss; bettermoments for moment-map evaluation (Sec. 9)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    # find the line-emission data dir under /kaggle/input (contains run_* subfolders)
    hits = glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

## 0b. Pull latest code (re-run anytime ΓÇö NO kernel restart needed)

After I push new changes to the `line-emission` branch, just **re-run this one cell** to fetch them
and hot-reload the `src/` modules. Then re-run the import cell below. No "Restart & Run All" required.

In [ ]:
# Pull latest from the line-emission branch and hot-reload src/ (no kernel restart).
# Self-contained: works even if the bootstrap cell hasn't run in this kernel.
import os, sys, subprocess

ON_KAGGLE = os.path.exists('/kaggle')
REPO = '/kaggle/working/EXXA'

if ON_KAGGLE and os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', 'line-emission'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/line-emission'], check=True)
    print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                         capture_output=True, text=True).stdout.strip())
elif ON_KAGGLE:
    print('repo not cloned yet ΓÇö run the bootstrap cell (0.) first')

# drop cached project modules so the next `import` picks up the freshly pulled code
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
print('src.* cleared from module cache ΓÇö now re-run the imports cell below.')

## 1. Imports, device, config


In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.models.unet import DenoisingUNet
from src.utils.losses import HybridLoss
from pytorch_msssim import ssim as ssim_torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU  = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device, '| GPUs:', N_GPU,
      '->', [torch.cuda.get_device_name(i) for i in range(N_GPU)] if N_GPU else 'cpu')

TARGET_SIZE   = 256     # test 256 first; raise to 300 only if VRAM allows
N_SAMPLES     = 50      # channels per cube
EPOCHS        = 30
LR            = 1e-3
# Start big to fill the GPU(s); the probe below shrinks on OOM. On T4x2 (DataParallel)
# this is split across both cards (e.g. 32 -> 16 per GPU). On 1 GPU it auto-drops to fit.
BATCH_SIZE    = 32

## 2. Cube-level split (3 RunID groups held out for inference only)


In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)

## 3. Datasets + DataLoaders (full-image 256x256, per-channel norm)


In [ ]:
train_ds = FITSChannelDataset(train_cubes, n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED)
val_ds   = FITSChannelDataset(val_cubes,   n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED)
print('train items:', len(train_ds), '| val items:', len(val_ds))

def make_loaders(bs):
    nw = 4 if ON_KAGGLE else 0   # FITS I/O is the bottleneck; more workers help a lot on Kaggle
    return (DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=nw, pin_memory=True),
            DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True))
train_loader, val_loader = make_loaders(BATCH_SIZE)

## 4. Model ΓÇö DenoisingUNet (last week's architecture, full image; t=0, sigmoid output)


In [ ]:
model = DenoisingUNet(str(device))
print('DenoisingUNet params:', f'{sum(p.numel() for p in model.parameters()):,}')
with torch.no_grad():
    o = model(torch.randn(1,1,TARGET_SIZE,TARGET_SIZE,device=device),
              torch.zeros(1,dtype=torch.long,device=device))
print('forward (1,1,%d,%d) -> %s' % (TARGET_SIZE, TARGET_SIZE, tuple(o.shape)))

## 5. Train ΓÇö 30 epochs, HybridLoss, Adam lr=1e-3

**Multi-GPU:** if more than one GPU is visible (Kaggle **T4 x2**), the model is wrapped in
`nn.DataParallel`, so each batch is split across both cards. Checkpoints are saved from the
*unwrapped* model, so they load fine on a single GPU later.

If the chosen batch OOMs at 256, the probe **reduces batch size first** (32->16->8...), keeping the
image at 256 ΓÇö never shrinking resolution before batch. The batch actually used is printed.

In [ ]:
criterion = HybridLoss(alpha=0.8, beta=0.2)

def core(m):
    """Unwrapped model (for state_dict / portable checkpoints)."""
    return m.module if isinstance(m, torch.nn.DataParallel) else m

def build_model_opt():
    m = DenoisingUNet(str(device))
    if N_GPU > 1:                      # use BOTH T4s
        m = torch.nn.DataParallel(m)
    return m, torch.optim.Adam(m.parameters(), lr=LR)

# OOM-safe batch probe (256px fixed; shrink batch only, never resolution)
def probe_batch(bs0):
    bs = bs0
    while bs >= 1:
        try:
            m, opt = build_model_opt()
            tl, _ = make_loaders(bs)
            d, c = next(iter(tl)); d, c = d.to(device), c.to(device)
            t = torch.zeros(d.size(0), dtype=torch.long, device=device)
            loss = criterion(m(d, t), c)[0]
            loss.backward()
            del m, opt, loss, d, c; torch.cuda.empty_cache()
            return bs
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower(): raise
            torch.cuda.empty_cache(); bs //= 2
            print(f'[OOM] reducing batch -> {bs} (image stays {TARGET_SIZE})')
    raise RuntimeError('does not fit even at batch 1')

BATCH_USED = probe_batch(BATCH_SIZE)
gpu_note = f'DataParallel x{N_GPU} (~{BATCH_USED // N_GPU}/GPU)' if N_GPU > 1 else f'single GPU'
print(f'BATCH SIZE USED: {BATCH_USED} at {TARGET_SIZE}x{TARGET_SIZE}  [{gpu_note}]')
train_loader, val_loader = make_loaders(BATCH_USED)

In [ ]:
model, optimizer = build_model_opt()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

CKPT = '../results/checkpoints/unet_line_emission_best.pth'
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
best_val, best_epoch, best_state = float('inf'), -1, None
tr_hist, va_hist = [], []

hdr = f"{'ep':>3} | {'tr_tot':>8} {'tr_mse':>8} {'tr_ssim':>8} | {'va_tot':>8} {'va_mse':>8} {'va_ssim':>8} | {'lr':>8}"
print(hdr); print('-'*len(hdr))

def run_epoch(loader, train):
    model.train(train)
    tot=mse=ssl=0.0; n=0
    torch.set_grad_enabled(train)
    for d, c in loader:
        d, c = d.to(device), c.to(device)
        t = torch.zeros(d.size(0), dtype=torch.long, device=device)
        pred = model(d, t)          # DataParallel splits the batch across GPUs
        total, m, s = criterion(pred, c)
        if train:
            optimizer.zero_grad(); total.backward(); optimizer.step()
        bs=d.size(0); tot+=total.item()*bs; mse+=m.item()*bs; ssl+=s.item()*bs; n+=bs
    torch.set_grad_enabled(True)
    return tot/n, mse/n, ssl/n

for ep in range(1, EPOCHS+1):
    t0=time.time()
    tr=run_epoch(train_loader, True)
    va=run_epoch(val_loader, False)
    scheduler.step(va[0]); tr_hist.append(tr[0]); va_hist.append(va[0])
    mark=''
    if va[0] < best_val:
        best_val, best_epoch = va[0], ep
        # save the UNWRAPPED state_dict (no 'module.' prefix -> loads on single GPU later)
        best_state={k:v.clone() for k,v in core(model).state_dict().items()}
        torch.save({'epoch':ep,'model_state_dict':best_state,'val_loss':best_val,
                    'alpha':0.8,'beta':0.2,'arch':'DenoisingUNet','target_size':TARGET_SIZE,
                    'batch_size':BATCH_USED}, CKPT)
        mark=' *best'
    print(f"{ep:>3} | {tr[0]:>8.4f} {tr[1]:>8.4f} {tr[2]:>8.4f} | "
          f"{va[0]:>8.4f} {va[1]:>8.4f} {va[2]:>8.4f} | {optimizer.param_groups[0]['lr']:>8.1e}"
          f"  {time.time()-t0:.0f}s{mark}")

print(f'\nbest val total {best_val:.4f} @ epoch {best_epoch}; checkpoint -> {CKPT}')
if best_state: core(model).load_state_dict(best_state)

## 6. Loss curve


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1,len(tr_hist)+1), tr_hist, marker='o', ms=3, label='train')
plt.plot(range(1,len(va_hist)+1), va_hist, marker='s', ms=3, label='val')
plt.axvline(best_epoch, color='gray', ls=':'); plt.scatter([best_epoch],[best_val],color='#E8715A',zorder=5)
plt.xlabel('epoch'); plt.ylabel('Hybrid loss'); plt.title(f'Line-emission U-Net ({TARGET_SIZE}px, batch {BATCH_USED})')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/unet_line_emission_loss.png', dpi=140); plt.show()

## 7. Validation metrics ΓÇö PSNR / SSIM / MSE


In [ ]:
model.eval()
psnrs, ssims, mses = [], [], []
with torch.no_grad():
    for d, c in val_loader:
        d, c = d.to(device), c.to(device)
        t = torch.zeros(d.size(0), dtype=torch.long, device=device)
        pred = model(d, t).clamp(0,1)
        mse = torch.mean((pred-c)**2, dim=(1,2,3))
        psnrs += (10*torch.log10(1.0/torch.clamp(mse,min=1e-10))).cpu().tolist()
        ssims += ssim_torch(pred, c, data_range=1.0, size_average=False).cpu().tolist()
        mses  += mse.cpu().tolist()
print(f'Validation ({len(mses)} channels):  PSNR {np.mean(psnrs):.4f} dB | '
      f'SSIM {np.mean(ssims):.4f} | MSE {np.mean(mses):.6f}')

## 8. Visualize 5 random validation channels ΓÇö dirty | denoised | clean


In [ ]:
import random
idxs = random.Random(SEED).sample(range(len(val_ds)), 5)
fig, ax = plt.subplots(5, 3, figsize=(10, 16))
cols=['dirty','U-Net denoised','clean GT']
for k, t in enumerate(cols): ax[0,k].set_title(t, fontweight='bold')
model.eval()
with torch.no_grad():
    for r, ix in enumerate(idxs):
        d, c = val_ds[ix]
        t0 = torch.zeros(1, dtype=torch.long, device=device)
        pred = model(d[None].to(device), t0)[0,0].cpu().numpy()
        ci, ch = val_ds.index[ix]
        for col, im in enumerate([d[0].numpy(), pred, c[0].numpy()]):
            ax[r,col].imshow(np.clip(im,0,1), cmap='inferno'); ax[r,col].axis('off')
        ax[r,0].set_ylabel(f'{val_ds.cube_paths[ci][2]}\nch {ch}', fontsize=8)
fig.suptitle('Line-emission U-Net ΓÇö validation channels', fontweight='bold', y=0.995)
plt.tight_layout()
os.makedirs('../experiments', exist_ok=True)
plt.savefig('../experiments/line_emission_unet_comparison.png', dpi=140); plt.show()
print('saved -> experiments/line_emission_unet_comparison.png')

## 9. Moment maps on a held-out cube (scientific evaluation target)

Moment maps are the real scientific product (mentor). Here we generate Moment 0/1/2 for a
**held-out** cube's clean and dirty versions with `bettermoments`, to confirm the package works and
to set up the end-to-end test. The eventual goal: denoise every channel of a held-out cube, rebuild
it, and show the **denoised** moment maps recover the clean kinematics (M1/M2) from the dirty ones.

In [ ]:
from src.evaluation.moment_maps import generate_moment_maps

# pick a held-out cube (inference-only) and its clean/dirty FITS
ho = holdout_cubes[0]
print('held-out cube:', ho['folder'])

c0, c1, c2 = generate_moment_maps(ho['clean'])
d0, d1, d2 = generate_moment_maps(ho['dirty'])

names = ['Moment 0 (intensity)', 'Moment 1 (velocity)', 'Moment 2 (dispersion)']
cmaps = ['inferno', 'RdBu_r', 'viridis']
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for col, (cm, dm) in enumerate([(c0, d0), (c1, d1), (c2, d2)]):
    vmin = float(np.nanmin([np.nanmin(cm), np.nanmin(dm)]))
    vmax = float(np.nanmax([np.nanmax(cm), np.nanmax(dm)]))
    for row, m in [(0, cm), (1, dm)]:
        im = ax[row, col].imshow(m, origin='lower', cmap=cmaps[col], vmin=vmin, vmax=vmax)
        ax[row, col].axis('off'); fig.colorbar(im, ax=ax[row, col], fraction=0.046, pad=0.04)
    ax[0, col].set_title(names[col])
fig.text(0.02, 0.74, 'CLEAN', fontsize=13, fontweight='bold', rotation=90, va='center')
fig.text(0.02, 0.26, 'DIRTY', fontsize=13, fontweight='bold', rotation=90, va='center')
fig.suptitle(f"Moment maps: clean vs dirty ΓÇö {ho['folder']}", fontweight='bold', fontsize=14)
plt.tight_layout(rect=[0.03, 0, 1, 0.97])
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/moment_maps_holdout.png', dpi=140); plt.show()

for nm, cl, di in [('M0', c0, d0), ('M1', c1, d1), ('M2', c2, d2)]:
    diff = np.nanmean(np.abs(cl - di)); denom = np.nanmax(np.abs(cl)) or 1.0
    print(f'{nm}: mean|diff| {diff:.4g} ({100*diff/denom:.1f}% of clean max)')

## 10. End-to-end: denoise a full held-out cube -> moment-map recovery

The real scientific test. Take a **held-out** cube (never in train/val), denoise **every one of its
~201 channels** with the trained U-Net, reassemble a full 600x600 cube, and compare moment maps of
**clean vs dirty vs denoised**. If denoising worked, the denoised moment maps should be **closer to
clean than the dirty ones** (smaller mean|diff|), especially M1/M2 (the kinematics).

**Correctness notes:**
- **Per-channel un-normalization (no leakage):** each channel is min-max normalized to [0,1] by its
  *own dirty* (min,max) before the model; the [0,1] output is un-normalized with that **same
  per-channel (min,max)** ΓÇö never a global cube min/max, never the clean cube's scale. The code below
  prints explicit confirmation.
- **Resolution:** the model only ever saw 256x256, so each denoised 600x600 channel is a 256x256
  prediction upsampled back up. The comparison is fair (clean/dirty/denoised all collapsed at 600x600),
  but the denoised output's *effective* resolution is lower than the data's native 600x600.
- `HOLDOUT_PICK` selects which held-out cube to use ΓÇö try others to find one whose clean M1 shows a
  clear non-Keplerian **kink** (the planet signature in Jason's papers) for a stronger story.


In [ ]:
import torch.nn.functional as F
from astropy.io import fits

CKPT = '../results/checkpoints/unet_line_emission_best.pth'
HOLDOUT_PICK = 0                      # change to try other held-out cubes (look for an M1 'kink')
ho = holdout_cubes[HOLDOUT_PICK]
print('Held-out cube (never trained/validated):', ho['folder'])

# load trained weights into a fresh, unwrapped model -> robust across kernel restarts
ckpt = torch.load(CKPT, map_location=device)
infer_net = DenoisingUNet(str(device))
infer_net.load_state_dict(ckpt['model_state_dict'])
infer_net.eval()
print('loaded checkpoint: epoch', ckpt.get('epoch'), '| val_loss', round(float(ckpt.get('val_loss', float('nan'))), 4))

# full dirty cube, ALL channels
with fits.open(ho['dirty'], memmap=False) as hdul:
    dirty_cube = np.ascontiguousarray(hdul[0].data).astype(np.float32)   # (C, 600, 600)
    dirty_hdr  = hdul[0].header.copy()
C, H, W = dirty_cube.shape
print('cube shape:', dirty_cube.shape)

# per-channel (min,max) FROM THE DIRTY INPUT ΓÇö stored and reused for un-normalization
los = dirty_cube.reshape(C, -1).min(axis=1)
his = dirty_cube.reshape(C, -1).max(axis=1)
rng = his - los
norm = np.zeros_like(dirty_cube)
nz = rng > 0
norm[nz] = (dirty_cube[nz] - los[nz, None, None]) / rng[nz, None, None]

# batched: 600 -> down 256 -> model -> up 600 -> un-normalize with the SAME per-channel (lo,hi)
denoised_cube = np.empty_like(dirty_cube)
BS = 32
with torch.no_grad():
    for s in range(0, C, BS):
        t = torch.from_numpy(norm[s:s+BS])[:, None].float().to(device)           # (b,1,600,600)
        t256 = F.interpolate(t, size=(TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
        tz = torch.zeros(t256.size(0), dtype=torch.long, device=device)
        out = infer_net(t256, tz)                                 # [0,1] (b,1,256,256)
        out600 = F.interpolate(out, size=(H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
        for k in range(out600.shape[0]):
            ch = s + k
            denoised_cube[ch] = (out600[k] * rng[ch] + los[ch]) if rng[ch] > 0 else np.full((H, W), los[ch], np.float32)

# explicit confirmation that un-normalization was PER-CHANNEL (not global)
print('\nUN-NORMALIZATION CONFIRMATION:')
print('  used each channel\'s OWN dirty (min,max) -> per-channel, NOT a global cube min/max, NOT clean scale.')
print(f'  e.g. ch 100 (min,max)=({los[100]:.5g}, {his[100]:.5g}); ch 50=({los[50]:.5g}, {his[50]:.5g})')
print(f'  {len(np.unique(np.round(rng, 8)))} distinct per-channel range-widths across {C} channels (global would be 1)')

# save denoised cube with the ORIGINAL dirty header (correct velocity axis for bettermoments)
out_fits = '../results/denoised_cube.fits'
os.makedirs('../results', exist_ok=True)
fits.writeto(out_fits, denoised_cube.astype(np.float32), header=dirty_hdr, overwrite=True)
print('\nsaved denoised cube ->', out_fits, '| shape', denoised_cube.shape)

In [ ]:
from src.evaluation.moment_maps import generate_moment_maps

c0, c1, c2 = generate_moment_maps(ho['clean'])     # clean cube
d0, d1, d2 = generate_moment_maps(ho['dirty'])     # original dirty cube
n0, n1, n2 = generate_moment_maps(out_fits)        # our denoised cube

rows  = [('clean', (c0, c1, c2)), ('dirty', (d0, d1, d2)), ('denoised', (n0, n1, n2))]
names = ['Moment 0 (intensity)', 'Moment 1 (velocity)', 'Moment 2 (dispersion)']
cmaps = ['inferno', 'RdBu_r', 'viridis']

fig, ax = plt.subplots(3, 3, figsize=(15, 15))
for col in range(3):
    allcol = [rows[r][1][col] for r in range(3)]               # shared scale per column
    vmin = float(np.nanmin([np.nanmin(a) for a in allcol]))
    vmax = float(np.nanmax([np.nanmax(a) for a in allcol]))
    for r in range(3):
        im = ax[r, col].imshow(rows[r][1][col], origin='lower', cmap=cmaps[col], vmin=vmin, vmax=vmax)
        ax[r, col].axis('off'); fig.colorbar(im, ax=ax[r, col], fraction=0.046, pad=0.04)
    ax[0, col].set_title(names[col], fontsize=12)
for r, (lbl, _) in enumerate(rows):
    ax[r, 0].text(-0.08, 0.5, lbl.upper(), transform=ax[r, 0].transAxes, fontsize=13,
                  fontweight='bold', rotation=90, va='center', ha='right')
fig.suptitle(f"Moment maps: clean vs dirty vs denoised ΓÇö {ho['folder']}", fontweight='bold', fontsize=15)
plt.tight_layout(rect=[0.02, 0, 1, 0.97])
plt.savefig('../results/moment_maps_denoised_comparison.png', dpi=140); plt.show()

# mean|diff| vs clean (finite overlap) for dirty and denoised, per moment
def mdiff(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.nanmean(np.abs(a[m] - b[m])))

print(f"\nHeld-out cube: {ho['folder']}")
print(f"{'moment':<8}{'dirty-vs-clean':>16}{'denoised-vs-clean':>20}{'improvement':>14}")
print('-' * 58)
for nm, cl, di, no in [('M0', c0, d0, n0), ('M1', c1, d1, n1), ('M2', c2, d2, n2)]:
    dd, nn = mdiff(cl, di), mdiff(cl, no)
    imp = 100 * (1 - nn / dd) if dd > 0 else float('nan')
    print(f"{nm:<8}{dd:>16.5g}{nn:>20.5g}{imp:>13.1f}%")
print("(positive improvement = denoised moment map is closer to clean than dirty was)")

In [ ]:
# --- DIAGNOSTIC: channel 100, dirty | denoised | clean, PHYSICAL (un-normalized) values ---
from astropy.io import fits
import numpy as np, matplotlib.pyplot as plt

CH = 100
with fits.open(ho['dirty'], memmap=False) as h:  dch = np.asarray(h[0].data[CH], dtype=np.float32)
with fits.open(ho['clean'], memmap=False) as h:  cch = np.asarray(h[0].data[CH], dtype=np.float32)
with fits.open(out_fits,   memmap=False) as h:  nch = np.asarray(h[0].data[CH], dtype=np.float32)

def rng(x): return f'[{np.nanmin(x):+.5g}, {np.nanmax(x):+.5g}]  mean {np.nanmean(x):+.5g}  std {np.nanstd(x):.5g}'
print(f'channel {CH} - PHYSICAL value ranges')
print(f'  dirty    {rng(dch)}')
print(f'  denoised {rng(nch)}')
print(f'  clean    {rng(cch)}')
print(f'\n  ratio denoised_max/clean_max = {np.nanmax(nch)/(np.nanmax(cch) or 1):.3g}   (~1 good; off = scale bug)')

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, im, t in zip(ax[:3], [dch, nch, cch], ['dirty','denoised','clean']):
    p = a.imshow(im, origin='lower', cmap='inferno'); a.set_title(f'{t} (own scale)'); a.axis('off')
    fig.colorbar(p, ax=a, fraction=0.046, pad=0.04)
vmin = float(min(np.nanmin(nch), np.nanmin(cch))); vmax = float(max(np.nanmax(nch), np.nanmax(cch)))
p = ax[3].imshow(nch, origin='lower', cmap='inferno', vmin=vmin, vmax=vmax)
ax[3].set_title('denoised (SHARED clean scale)'); ax[3].axis('off'); fig.colorbar(p, ax=ax[3], fraction=0.046, pad=0.04)
fig.suptitle(f"Channel {CH} diagnostic - {ho['folder']}", fontweight='bold'); plt.tight_layout(); plt.show()

---

# 06 ΓÇö Continuum-subtracted Line Emission (U-Net)

_Appended from 06-unet-line-emission-continuum.ipynb. Cells below are self-contained; re-run setup as needed._


# Line Emission Denoising ├óΓé¼ΓÇ¥ Full-Image U-Net (Week-5 pivot)

Per the 2026-06-18 mentor pivot: **full-image** denoising of line-emission velocity channels
(NO patches), using **last week's U-Net** (`DenoisingUNet`, NOT the DDPM), trained channel-by-channel.

> **Continuum-subtraction variant (mentor, 2026-06-27):** identical to `05` except each channel has the
> static continuum removed first ΓÇö the mean of the first/last `CONTINUUM_N` line-free channels is
> subtracted from every channel (dataset + Section 10), isolating the line emission. A/B against `05`.

- Dataset: line-emission FITS cubes, (201, 600, 600), split at the **cube level** (3 RunID groups
  held out for inference only).
- Channels sampled per cube via the Gaussian sampler (center 100, ~75% in [50,150]).
- Each channel downsampled to **256├âΓÇö256**, per-channel min-max normalised to [0,1].
- Loss: `HybridLoss(0.8, 0.2)` = 0.8├é┬╖MSE + 0.2├é┬╖(1├ó╦åΓÇÖSSIM). Optimizer Adam lr=1e-3.

### Kaggle setup
GPU on, Internet on, *Add Input* ├óΓÇáΓÇÖ your line-emission Dataset (FITS cubes). The bootstrap finds it
under `/kaggle/input/` and points the split at it.


## 0. Bootstrap (clone repo for src/, locate data)


In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch','line-emission','--depth','1','https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin','line-emission'], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/line-emission'], check=True)
    # pytorch-msssim for the SSIM loss; bettermoments for moment-map evaluation (Sec. 9)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    # find the line-emission data dir under /kaggle/input (contains run_* subfolders)
    hits = glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

## 0b. Pull latest code (re-run anytime ├óΓé¼ΓÇ¥ NO kernel restart needed)

After I push new changes to the `line-emission` branch, just **re-run this one cell** to fetch them
and hot-reload the `src/` modules. Then re-run the import cell below. No "Restart & Run All" required.

In [ ]:
# Pull latest from the line-emission branch and hot-reload src/ (no kernel restart).
# Self-contained: works even if the bootstrap cell hasn't run in this kernel.
import os, sys, subprocess

ON_KAGGLE = os.path.exists('/kaggle')
REPO = '/kaggle/working/EXXA'

if ON_KAGGLE and os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', 'line-emission'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/line-emission'], check=True)
    print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                         capture_output=True, text=True).stdout.strip())
elif ON_KAGGLE:
    print('repo not cloned yet ├óΓé¼ΓÇ¥ run the bootstrap cell (0.) first')

# drop cached project modules so the next `import` picks up the freshly pulled code
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
print('src.* cleared from module cache ├óΓé¼ΓÇ¥ now re-run the imports cell below.')

## 1. Imports, device, config


In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.models.unet import DenoisingUNet
from src.utils.losses import HybridLoss
from pytorch_msssim import ssim as ssim_torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU  = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device, '| GPUs:', N_GPU,
      '->', [torch.cuda.get_device_name(i) for i in range(N_GPU)] if N_GPU else 'cpu')

TARGET_SIZE   = 256     # test 256 first; raise to 300 only if VRAM allows
N_SAMPLES     = 50      # channels per cube
EPOCHS        = 30
LR            = 1e-3
# Start big to fill the GPU(s); the probe below shrinks on OOM. On T4x2 (DataParallel)
# this is split across both cards (e.g. 32 -> 16 per GPU). On 1 GPU it auto-drops to fit.
BATCH_SIZE    = 32

## 2. Cube-level split (3 RunID groups held out for inference only)


In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)

## 3. Datasets + DataLoaders (full-image 256x256, per-channel norm)


In [ ]:
SUBTRACT_CONTINUUM = True     # <-- the only switch vs notebook 05; subtract line-free continuum
CONTINUUM_N        = 5        # channels at each end averaged for the continuum estimate
train_ds = FITSChannelDataset(train_cubes, n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
val_ds   = FITSChannelDataset(val_cubes,   n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
print('train items:', len(train_ds), '| val items:', len(val_ds))

def make_loaders(bs):
    nw = 4 if ON_KAGGLE else 0   # FITS I/O is the bottleneck; more workers help a lot on Kaggle
    return (DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=nw, pin_memory=True),
            DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True))
train_loader, val_loader = make_loaders(BATCH_SIZE)

## 4. Model ├óΓé¼ΓÇ¥ DenoisingUNet (last week's architecture, full image; t=0, sigmoid output)


In [ ]:
model = DenoisingUNet(str(device))
print('DenoisingUNet params:', f'{sum(p.numel() for p in model.parameters()):,}')
with torch.no_grad():
    o = model(torch.randn(1,1,TARGET_SIZE,TARGET_SIZE,device=device),
              torch.zeros(1,dtype=torch.long,device=device))
print('forward (1,1,%d,%d) -> %s' % (TARGET_SIZE, TARGET_SIZE, tuple(o.shape)))

## 5. Train ├óΓé¼ΓÇ¥ 30 epochs, HybridLoss, Adam lr=1e-3

**Multi-GPU:** if more than one GPU is visible (Kaggle **T4 x2**), the model is wrapped in
`nn.DataParallel`, so each batch is split across both cards. Checkpoints are saved from the
*unwrapped* model, so they load fine on a single GPU later.

If the chosen batch OOMs at 256, the probe **reduces batch size first** (32->16->8...), keeping the
image at 256 ├óΓé¼ΓÇ¥ never shrinking resolution before batch. The batch actually used is printed.

In [ ]:
criterion = HybridLoss(alpha=0.8, beta=0.2)

def core(m):
    """Unwrapped model (for state_dict / portable checkpoints)."""
    return m.module if isinstance(m, torch.nn.DataParallel) else m

def build_model_opt():
    m = DenoisingUNet(str(device))
    if N_GPU > 1:                      # use BOTH T4s
        m = torch.nn.DataParallel(m)
    return m, torch.optim.Adam(m.parameters(), lr=LR)

# OOM-safe batch probe (256px fixed; shrink batch only, never resolution)
def probe_batch(bs0):
    bs = bs0
    while bs >= 1:
        try:
            m, opt = build_model_opt()
            tl, _ = make_loaders(bs)
            d, c = next(iter(tl)); d, c = d.to(device), c.to(device)
            t = torch.zeros(d.size(0), dtype=torch.long, device=device)
            loss = criterion(m(d, t), c)[0]
            loss.backward()
            del m, opt, loss, d, c; torch.cuda.empty_cache()
            return bs
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower(): raise
            torch.cuda.empty_cache(); bs //= 2
            print(f'[OOM] reducing batch -> {bs} (image stays {TARGET_SIZE})')
    raise RuntimeError('does not fit even at batch 1')

BATCH_USED = probe_batch(BATCH_SIZE)
gpu_note = f'DataParallel x{N_GPU} (~{BATCH_USED // N_GPU}/GPU)' if N_GPU > 1 else f'single GPU'
print(f'BATCH SIZE USED: {BATCH_USED} at {TARGET_SIZE}x{TARGET_SIZE}  [{gpu_note}]')
train_loader, val_loader = make_loaders(BATCH_USED)

In [ ]:
model, optimizer = build_model_opt()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

CKPT = '../results/checkpoints/unet_line_emission_continuum_best.pth'
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
best_val, best_epoch, best_state = float('inf'), -1, None
tr_hist, va_hist = [], []

hdr = f"{'ep':>3} | {'tr_tot':>8} {'tr_mse':>8} {'tr_ssim':>8} | {'va_tot':>8} {'va_mse':>8} {'va_ssim':>8} | {'lr':>8}"
print(hdr); print('-'*len(hdr))

def run_epoch(loader, train):
    model.train(train)
    tot=mse=ssl=0.0; n=0
    torch.set_grad_enabled(train)
    for d, c in loader:
        d, c = d.to(device), c.to(device)
        t = torch.zeros(d.size(0), dtype=torch.long, device=device)
        pred = model(d, t)          # DataParallel splits the batch across GPUs
        total, m, s = criterion(pred, c)
        if train:
            optimizer.zero_grad(); total.backward(); optimizer.step()
        bs=d.size(0); tot+=total.item()*bs; mse+=m.item()*bs; ssl+=s.item()*bs; n+=bs
    torch.set_grad_enabled(True)
    return tot/n, mse/n, ssl/n

for ep in range(1, EPOCHS+1):
    t0=time.time()
    tr=run_epoch(train_loader, True)
    va=run_epoch(val_loader, False)
    scheduler.step(va[0]); tr_hist.append(tr[0]); va_hist.append(va[0])
    mark=''
    if va[0] < best_val:
        best_val, best_epoch = va[0], ep
        # save the UNWRAPPED state_dict (no 'module.' prefix -> loads on single GPU later)
        best_state={k:v.clone() for k,v in core(model).state_dict().items()}
        torch.save({'epoch':ep,'model_state_dict':best_state,'val_loss':best_val,
                    'alpha':0.8,'beta':0.2,'arch':'DenoisingUNet','target_size':TARGET_SIZE,
                    'batch_size':BATCH_USED}, CKPT)
        mark=' *best'
    print(f"{ep:>3} | {tr[0]:>8.4f} {tr[1]:>8.4f} {tr[2]:>8.4f} | "
          f"{va[0]:>8.4f} {va[1]:>8.4f} {va[2]:>8.4f} | {optimizer.param_groups[0]['lr']:>8.1e}"
          f"  {time.time()-t0:.0f}s{mark}")

print(f'\nbest val total {best_val:.4f} @ epoch {best_epoch}; checkpoint -> {CKPT}')
if best_state: core(model).load_state_dict(best_state)

## 6. Loss curve


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1,len(tr_hist)+1), tr_hist, marker='o', ms=3, label='train')
plt.plot(range(1,len(va_hist)+1), va_hist, marker='s', ms=3, label='val')
plt.axvline(best_epoch, color='gray', ls=':'); plt.scatter([best_epoch],[best_val],color='#E8715A',zorder=5)
plt.xlabel('epoch'); plt.ylabel('Hybrid loss'); plt.title(f'Line-emission U-Net ({TARGET_SIZE}px, batch {BATCH_USED})')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/unet_line_emission_loss.png', dpi=140); plt.show()

## 7. Validation metrics ├óΓé¼ΓÇ¥ PSNR / SSIM / MSE


In [ ]:
model.eval()
psnrs, ssims, mses = [], [], []
with torch.no_grad():
    for d, c in val_loader:
        d, c = d.to(device), c.to(device)
        t = torch.zeros(d.size(0), dtype=torch.long, device=device)
        pred = model(d, t).clamp(0,1)
        mse = torch.mean((pred-c)**2, dim=(1,2,3))
        psnrs += (10*torch.log10(1.0/torch.clamp(mse,min=1e-10))).cpu().tolist()
        ssims += ssim_torch(pred, c, data_range=1.0, size_average=False).cpu().tolist()
        mses  += mse.cpu().tolist()
print(f'Validation ({len(mses)} channels):  PSNR {np.mean(psnrs):.4f} dB | '
      f'SSIM {np.mean(ssims):.4f} | MSE {np.mean(mses):.6f}')

## 8. Visualize 5 random validation channels ├óΓé¼ΓÇ¥ dirty | denoised | clean


In [ ]:
import random
idxs = random.Random(SEED).sample(range(len(val_ds)), 5)
fig, ax = plt.subplots(5, 3, figsize=(10, 16))
cols=['dirty','U-Net denoised','clean GT']
for k, t in enumerate(cols): ax[0,k].set_title(t, fontweight='bold')
model.eval()
with torch.no_grad():
    for r, ix in enumerate(idxs):
        d, c = val_ds[ix]
        t0 = torch.zeros(1, dtype=torch.long, device=device)
        pred = model(d[None].to(device), t0)[0,0].cpu().numpy()
        ci, ch = val_ds.index[ix]
        for col, im in enumerate([d[0].numpy(), pred, c[0].numpy()]):
            ax[r,col].imshow(np.clip(im,0,1), cmap='inferno'); ax[r,col].axis('off')
        ax[r,0].set_ylabel(f'{val_ds.cube_paths[ci][2]}\nch {ch}', fontsize=8)
fig.suptitle('Line-emission U-Net ├óΓé¼ΓÇ¥ validation channels', fontweight='bold', y=0.995)
plt.tight_layout()
os.makedirs('../experiments', exist_ok=True)
plt.savefig('../experiments/line_emission_unet_comparison.png', dpi=140); plt.show()
print('saved -> experiments/line_emission_unet_comparison.png')

## 9. Moment maps on a held-out cube (scientific evaluation target)

Moment maps are the real scientific product (mentor). Here we generate Moment 0/1/2 for a
**held-out** cube's clean and dirty versions with `bettermoments`, to confirm the package works and
to set up the end-to-end test. The eventual goal: denoise every channel of a held-out cube, rebuild
it, and show the **denoised** moment maps recover the clean kinematics (M1/M2) from the dirty ones.

In [ ]:
from src.evaluation.moment_maps import generate_moment_maps

# pick a held-out cube (inference-only) and its clean/dirty FITS
ho = holdout_cubes[0]
print('held-out cube:', ho['folder'])

c0, c1, c2 = generate_moment_maps(ho['clean'])
d0, d1, d2 = generate_moment_maps(ho['dirty'])

names = ['Moment 0 (intensity)', 'Moment 1 (velocity)', 'Moment 2 (dispersion)']
cmaps = ['inferno', 'RdBu_r', 'viridis']
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for col, (cm, dm) in enumerate([(c0, d0), (c1, d1), (c2, d2)]):
    vmin = float(np.nanmin([np.nanmin(cm), np.nanmin(dm)]))
    vmax = float(np.nanmax([np.nanmax(cm), np.nanmax(dm)]))
    for row, m in [(0, cm), (1, dm)]:
        im = ax[row, col].imshow(m, origin='lower', cmap=cmaps[col], vmin=vmin, vmax=vmax)
        ax[row, col].axis('off'); fig.colorbar(im, ax=ax[row, col], fraction=0.046, pad=0.04)
    ax[0, col].set_title(names[col])
fig.text(0.02, 0.74, 'CLEAN', fontsize=13, fontweight='bold', rotation=90, va='center')
fig.text(0.02, 0.26, 'DIRTY', fontsize=13, fontweight='bold', rotation=90, va='center')
fig.suptitle(f"Moment maps: clean vs dirty ├óΓé¼ΓÇ¥ {ho['folder']}", fontweight='bold', fontsize=14)
plt.tight_layout(rect=[0.03, 0, 1, 0.97])
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/moment_maps_holdout.png', dpi=140); plt.show()

for nm, cl, di in [('M0', c0, d0), ('M1', c1, d1), ('M2', c2, d2)]:
    diff = np.nanmean(np.abs(cl - di)); denom = np.nanmax(np.abs(cl)) or 1.0
    print(f'{nm}: mean|diff| {diff:.4g} ({100*diff/denom:.1f}% of clean max)')

## 10. End-to-end: denoise a full held-out cube -> moment-map recovery

The real scientific test. Take a **held-out** cube (never in train/val), denoise **every one of its
~201 channels** with the trained U-Net, reassemble a full 600x600 cube, and compare moment maps of
**clean vs dirty vs denoised**. If denoising worked, the denoised moment maps should be **closer to
clean than the dirty ones** (smaller mean|diff|), especially M1/M2 (the kinematics).

**Correctness notes:**
- **Per-channel un-normalization (no leakage):** each channel is min-max normalized to [0,1] by its
  *own dirty* (min,max) before the model; the [0,1] output is un-normalized with that **same
  per-channel (min,max)** ├óΓé¼ΓÇ¥ never a global cube min/max, never the clean cube's scale. The code below
  prints explicit confirmation.
- **Resolution:** the model only ever saw 256x256, so each denoised 600x600 channel is a 256x256
  prediction upsampled back up. The comparison is fair (clean/dirty/denoised all collapsed at 600x600),
  but the denoised output's *effective* resolution is lower than the data's native 600x600.
- `HOLDOUT_PICK` selects which held-out cube to use ├óΓé¼ΓÇ¥ try others to find one whose clean M1 shows a
  clear non-Keplerian **kink** (the planet signature in Jason's papers) for a stronger story.


In [ ]:
import torch.nn.functional as F
from astropy.io import fits

CKPT = '../results/checkpoints/unet_line_emission_continuum_best.pth'
HOLDOUT_PICK = 0                      # change to try other held-out cubes (look for an M1 'kink')
ho = holdout_cubes[HOLDOUT_PICK]
CONTINUUM_N = 5                       # channels at each end for the continuum estimate (match dataset)
print('Held-out cube (never trained/validated):', ho['folder'])

# load trained weights into a fresh, unwrapped model -> robust across kernel restarts
ckpt = torch.load(CKPT, map_location=device)
infer_net = DenoisingUNet(str(device))
infer_net.load_state_dict(ckpt['model_state_dict'])
infer_net.eval()
print('loaded checkpoint: epoch', ckpt.get('epoch'), '| val_loss', round(float(ckpt.get('val_loss', float('nan'))), 4))

# full dirty cube, ALL channels (RAW, before continuum subtraction)
with fits.open(ho['dirty'], memmap=False) as hdul:
    dirty_cube_raw = np.ascontiguousarray(hdul[0].data).astype(np.float32)   # (C, 600, 600)
    dirty_hdr      = hdul[0].header.copy()
C, H, W = dirty_cube_raw.shape
print('cube shape:', dirty_cube_raw.shape)

# CONTINUUM SUBTRACTION (mentor): 2D continuum = mean of first/last N line-free channels,
# subtracted from EVERY channel -> isolates line emission. Same op the dataset applied in training.
def continuum_of(cube, n):
    n = max(1, min(n, C // 2))
    return np.concatenate([cube[:n], cube[C - n:]], axis=0).mean(axis=0)
dcont = continuum_of(dirty_cube_raw, CONTINUUM_N)
dirty_cube = dirty_cube_raw - dcont[None]        # line-only dirty (the model input)
print('continuum subtracted (mean of first/last', CONTINUUM_N, 'ch): dcont range',
      [round(float(dcont.min()), 6), round(float(dcont.max()), 6)])

# per-channel (min,max) FROM the continuum-subtracted DIRTY -> stored for un-normalization
los = dirty_cube.reshape(C, -1).min(axis=1)
his = dirty_cube.reshape(C, -1).max(axis=1)
rng = his - los
norm = np.zeros_like(dirty_cube)
nz = rng > 0
norm[nz] = (dirty_cube[nz] - los[nz, None, None]) / rng[nz, None, None]

# batched: 600 -> down 256 -> model -> up 600 -> un-normalize with the SAME per-channel (lo,hi)
denoised_cube = np.empty_like(dirty_cube)
BS = 32
with torch.no_grad():
    for s in range(0, C, BS):
        t = torch.from_numpy(norm[s:s+BS])[:, None].float().to(device)           # (b,1,600,600)
        t256 = F.interpolate(t, size=(TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
        tz = torch.zeros(t256.size(0), dtype=torch.long, device=device)
        out = infer_net(t256, tz)                                                # linear (b,1,256,256)
        out600 = F.interpolate(out, size=(H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
        for k in range(out600.shape[0]):
            ch = s + k
            denoised_cube[ch] = (out600[k] * rng[ch] + los[ch]) if rng[ch] > 0 else np.full((H, W), los[ch], np.float32)

print()
print('CONTINUUM + UN-NORMALIZATION:')
print('  continuum-subtracted (first/last', CONTINUUM_N, 'ch) BEFORE per-channel shared dirty-scale norm.')
print('  decoded denoised cube is line-only (continuum removed); decode uses each per-channel OWN')
print('  continuum-subtracted dirty (min,max) -> invertible, same scale the clean target used.')

# save denoised (line-only) cube with the ORIGINAL dirty header (correct velocity axis)
out_fits = '../results/denoised_cube_continuum.fits'
os.makedirs('../results', exist_ok=True)
fits.writeto(out_fits, denoised_cube.astype(np.float32), header=dirty_hdr, overwrite=True)
print()
print('saved denoised cube ->', out_fits, '| shape', denoised_cube.shape)

In [ ]:
import bettermoments as bm
from src.evaluation.moment_maps import generate_moment_maps

# Fair comparison: clean & dirty references are ALSO continuum-subtracted (line-only), like the
# denoised cube. velax shared from the dirty cube header.
_, velax = bm.load_cube(ho['dirty'])
with fits.open(ho['clean'], memmap=False) as h:
    clean_raw = np.ascontiguousarray(h[0].data).astype(np.float32)
ccont = continuum_of(clean_raw, CONTINUUM_N)
clean_csub = clean_raw - ccont[None]             # clean line-only (its OWN continuum)
dirty_csub = dirty_cube                          # already continuum-subtracted in the cell above

c0, c1, c2 = generate_moment_maps(None, data_velax=(clean_csub, velax))   # clean  (line-only)
d0, d1, d2 = generate_moment_maps(None, data_velax=(dirty_csub, velax))   # dirty  (line-only)
n0, n1, n2 = generate_moment_maps(out_fits)                               # denoised (line-only)

rows  = [('clean', (c0, c1, c2)), ('dirty', (d0, d1, d2)), ('denoised', (n0, n1, n2))]
names = ['Moment 0 (intensity)', 'Moment 1 (velocity)', 'Moment 2 (dispersion)']
cmaps = ['inferno', 'RdBu_r', 'viridis']

fig, ax = plt.subplots(3, 3, figsize=(15, 15))
for col in range(3):
    allcol = [rows[r][1][col] for r in range(3)]               # shared scale per column
    vmin = float(np.nanmin([np.nanmin(a) for a in allcol]))
    vmax = float(np.nanmax([np.nanmax(a) for a in allcol]))
    for r in range(3):
        im = ax[r, col].imshow(rows[r][1][col], origin='lower', cmap=cmaps[col], vmin=vmin, vmax=vmax)
        ax[r, col].axis('off'); fig.colorbar(im, ax=ax[r, col], fraction=0.046, pad=0.04)
    ax[0, col].set_title(names[col], fontsize=12)
for r, (lbl, _) in enumerate(rows):
    ax[r, 0].text(-0.08, 0.5, lbl.upper(), transform=ax[r, 0].transAxes, fontsize=13,
                  fontweight='bold', rotation=90, va='center', ha='right')
fig.suptitle('Moment maps (continuum-subtracted): clean vs dirty vs denoised - ' + ho['folder'],
             fontweight='bold', fontsize=15)
plt.tight_layout(rect=[0.02, 0, 1, 0.97])
plt.savefig('../results/moment_maps_continuum_comparison.png', dpi=140); plt.show()

# mean|diff| vs clean (finite overlap) for dirty and denoised, per moment
def mdiff(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.nanmean(np.abs(a[m] - b[m])))

print()
print('Held-out cube (CONTINUUM-SUBTRACTED):', ho['folder'])
hdr = '{:<8}{:>16}{:>20}{:>14}'.format('moment', 'dirty-vs-clean', 'denoised-vs-clean', 'improvement')
print(hdr); print('-' * 58)
for nm, cl, di, no in [('M0', c0, d0, n0), ('M1', c1, d1, n1), ('M2', c2, d2, n2)]:
    dd, nn = mdiff(cl, di), mdiff(cl, no)
    imp = 100 * (1 - nn / dd) if dd > 0 else float('nan')
    print('{:<8}{:>16.5g}{:>20.5g}{:>13.1f}%'.format(nm, dd, nn, imp))
print('(positive improvement = denoised moment map is closer to clean than dirty was)')

In [ ]:
# --- DIAGNOSTIC: channel 100, dirty | denoised | clean, CONTINUUM-SUBTRACTED physical values ---
from astropy.io import fits
import numpy as np, matplotlib.pyplot as plt

CH = 100
with fits.open(ho['clean'], memmap=False) as h:
    clean_raw = np.ascontiguousarray(h[0].data).astype(np.float32)
ccont = continuum_of(clean_raw, CONTINUUM_N)
dch = dirty_cube_raw[CH] - dcont          # dirty line-only
cch = clean_raw[CH]      - ccont          # clean line-only
with fits.open(out_fits, memmap=False) as h:
    nch = np.asarray(h[0].data[CH], dtype=np.float32)   # denoised (already line-only)

def rng(x):
    return '[{:+.5g}, {:+.5g}]  mean {:+.5g}  std {:.5g}'.format(
        float(np.nanmin(x)), float(np.nanmax(x)), float(np.nanmean(x)), float(np.nanstd(x)))
print('channel', CH, '- CONTINUUM-SUBTRACTED physical value ranges')
print('  dirty   ', rng(dch))
print('  denoised', rng(nch))
print('  clean   ', rng(cch))
print('  ratio denoised_max/clean_max =',
      round(float(np.nanmax(nch) / (np.nanmax(cch) or 1)), 3), '  (~1 good; off = scale/overshoot)')

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, im, t in zip(ax[:3], [dch, nch, cch], ['dirty', 'denoised', 'clean']):
    p = a.imshow(im, origin='lower', cmap='inferno'); a.set_title(t + ' (own scale, line-only)'); a.axis('off')
    fig.colorbar(p, ax=a, fraction=0.046, pad=0.04)
vmin = float(min(np.nanmin(nch), np.nanmin(cch))); vmax = float(max(np.nanmax(nch), np.nanmax(cch)))
p = ax[3].imshow(nch, origin='lower', cmap='inferno', vmin=vmin, vmax=vmax)
ax[3].set_title('denoised (SHARED clean scale)'); ax[3].axis('off'); fig.colorbar(p, ax=ax[3], fraction=0.046, pad=0.04)
fig.suptitle('Channel ' + str(CH) + ' diagnostic (continuum-subtracted) - ' + ho['folder'], fontweight='bold')
plt.tight_layout(); plt.show()